[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Digital_Communications.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Digital Communications

How bits become waveforms and survive the trip back: modulation, matched filtering, synchronization, and OFDM — a working QAM link and a working multicarrier link, both built from scratch in NumPy.

## 1. Pre-requisites

- [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (complex exponentials, DFT).
- [Statistical Signal Processing](./Statistical_Signal_Processing.ipynb) S4 (matched filters).
- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) S4 for the capacity backdrop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Modulation: Bits → Waveforms* (~35 min)
**Goal:** map bits to constellation symbols; understand the energy/rate trade.
**Feeds into:** Session 2 (matched filter & eyes).

---

## 2. Constellations

💡 **Intuition.** A passband waveform $A\cos(2\pi f_c t + \phi)$ has two independent knobs — amplitude-on-cosine and amplitude-on-sine — so each symbol period carries a **complex number** $I + jQ$. A *constellation* is the alphabet of complex points you allow: QPSK uses 4 (2 bits/symbol), 16-QAM uses 16 (4 bits/symbol). More points ⇒ more bits per symbol ⇒ points closer together ⇒ noise flips them more easily. Modulation design is packing points on a power budget — with [capacity](../Intro_Math/Information_Theory/Information_Theory.ipynb) as the referee.

In [ ]:

# YOUR CODE HERE


**What just happened.** Two constellations at the **same 10 dB SNR** and the same average transmit power. QPSK's four clusters are widely separated with clear gaps; 16-QAM's sixteen clusters are visibly crowded, with neighbouring clouds beginning to touch. Every point of contact is a symbol error waiting to happen.

**The exchange rate is computable.** Both constellations are normalised to unit average energy — that is what `pts / np.sqrt((np.abs(pts)**2).mean())` enforces, and it is what makes the comparison fair. Under that budget, QPSK's points sit at radius 1 with a nearest-neighbour distance of $\sqrt{2} \approx 1.41$; 16-QAM's minimum distance is about **0.63**. So doubling the bits per symbol from 2 to 4 cost roughly a factor of 2.2 in minimum distance, which is about **7 dB** of noise margin. That is the price list for density, and it is the entire content of the session.

**Where the complex plane comes from.** A passband waveform $A\cos(2\pi f_c t + \phi)$ expands as $I\cos - Q\sin$, and cosine and sine are orthogonal over a symbol period — the [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) orthogonality from earlier in the track, earning its keep. So a carrier has two independent amplitude knobs, and each symbol genuinely carries a complex number. These axes are physical, not notational.

Seen that way, modulation is a **packing problem**: given a disc whose radius is set by your power budget, place as many points as possible while keeping them far enough apart to survive the noise. Everything else in constellation design follows from that.

**Neither alphabet is better — and that is why your Wi-Fi changes speed.** Each is optimal over a range of SNR, and [Shannon capacity](../Intro_Math/Information_Theory/Information_Theory.ipynb) sets how many bits per symbol a given SNR can support at all. Real links therefore use *adaptive modulation*: strong signal, use 256-QAM and enjoy the throughput; walk away from the router and the link steps down through 64-QAM, 16-QAM, QPSK, trading rate for margin as it goes. The two scatter plots above are two rungs of that ladder, and the router is choosing between them many times per second.

---
### 🕐 Session 2 of 4 — *Pulse Shaping, Matched Filters & Eye Diagrams* (~40 min)
**Goal:** put symbols on pulses without smearing neighbors; read link health from the eye.
**Builds on:** Session 1; [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 3 (synchronization).

---

## 3. From Symbols to Samples

💡 **Intuition.** You can't transmit points — you transmit *pulses* carrying points, and bandwidth limits force pulses to be long, overlapping their neighbors. The escape is **Nyquist pulses** (raised cosine): they may overlap everywhere *except at the sampling instants*, where every other pulse crosses zero — no inter-symbol interference *if you sample exactly on time*. The receiver applies the [matched filter](./Statistical_Signal_Processing.ipynb) for SNR, and the **eye diagram** — all symbol periods overlaid — shows your margins: vertical opening = noise margin, horizontal = timing margin.

In [ ]:
# Root-raised-cosine link, oversampled 8x

# YOUR CODE HERE


**What just happened.** A wide-open eye and a printed symbol error rate of **0.0000 over 500 symbols**. The eye is the real result; the zero is not a measurement, and it is worth being precise about why.

**Start with the number, because it is a trap.** Theoretical QPSK symbol error rate at 14 dB is about $5.4\times10^{-7}$. Across 500 symbols that predicts $2.7\times10^{-4}$ errors — so observing zero was essentially guaranteed before the cell ran, and it would have been equally guaranteed for a link 100× worse. To expect even one error you would need roughly **1.9 million symbols**. By the rule of three, zero errors in 500 trials justifies only a 95% upper bound of $3/500 = 0.006$, which sits four orders of magnitude above the true rate.

So "0.0000" reports the *experiment's* resolution, not the link's quality. Measuring a rare event requires a sample size scaled to its rarity, and BER simulations in practice run millions of symbols or use importance sampling for exactly this reason. A zero in an error-rate column should always prompt "how many trials?" before it prompts satisfaction.

**Now the eye, which does carry information.** Read it in three parts. The **vertical** opening at the red line is noise margin — how much noise can be added before traces cross the decision threshold. The **horizontal** width of the opening is timing margin — how far off the ideal instant you can sample and still decide correctly. And the **thickness** of the traces is residual ISI plus noise. A single picture, three diagnostics, which is why engineers look at an eye before they look at any number.

**Why the pulses can overlap and still work.** You cannot transmit a point, only a pulse carrying one, and bandwidth limits force pulses to be long — so they *do* overlap their neighbours, visibly, between the sampling instants. The escape is the **Nyquist criterion**: a raised-cosine pulse is 1 at its own sampling instant and exactly zero at every other symbol's. At the moment you sample, every neighbour contributes nothing. Zero ISI is engineered into the pulse shape rather than removed by the receiver.

**And the qualifier is the whole of Session 3.** No ISI *if you sample exactly on time*. Off-timing loses the zero crossings and every neighbour starts leaking in — which is precisely what the horizontal eye opening measures. Timing recovery is therefore a requirement, not a refinement.

**One design detail worth noticing.** Both ends use a *root*-raised-cosine, because RRC ∗ RRC = raised cosine. Splitting the Nyquist pulse across transmitter and receiver means the receive filter is *matched* to the transmitted pulse — maximising SNR by [Statistical SP](./Statistical_Signal_Processing.ipynb) S4 — while the end-to-end response still satisfies the zero-ISI condition. Two requirements satisfied by one factorisation, which is why you never see plain raised-cosine at the transmitter alone.

---
### 🕐 Session 3 of 4 — *Synchronization* (~35 min)
**Goal:** find the frame and the phase: correlation sync and the cost of being wrong.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (OFDM).

---

## 4. Where Does the Frame Start?

💡 **Intuition.** The receiver knows neither *when* symbols start nor *what phase* the oscillator drifted to. Both are solved with correlation: prepend a known **preamble**; the receiver slides it across the incoming stream, and the correlation peak marks the frame start ([matched filter](./Statistical_Signal_Processing.ipynb) again — detection in time). The *phase* of that same peak reveals the carrier phase offset — one correlation, two syncs. Every Wi-Fi packet begins exactly this way.

In [ ]:

# YOUR CODE HERE


**What just happened.** The receiver was handed a stream containing noise, then a frame at an unknown offset with an unknown phase rotation, then more noise. One correlation recovered **both** unknowns, and the payload decoded with **0 errors out of 400**.

**One correlation, two syncs — this is the elegant part.** The correlation *magnitude* peaks where the preamble aligns, which gives timing. The correlation *phase* at that peak gives the carrier phase offset, because
$$\sum_k (s_k e^{j\phi})\,\overline{s_k} = e^{j\phi}\sum_k |s_k|^2,$$
so rotating the whole frame by $\phi$ rotates the correlation peak by exactly $\phi$ and leaves the magnitude untouched. The phase estimate is free — it comes from a computation you were already performing for timing.

And the operation itself is one this curriculum keeps reusing: sliding a known template across data and taking the peak is the **matched filter** from [Statistical SP](./Statistical_Signal_Processing.ipynb) S4, the same operation as radar [pulse compression](./Radar_Signal_Processing.ipynb). Three workshops, three apparently unrelated jobs, one theorem.

**Why the preamble must be designed, not arbitrary.** It needs a sharp autocorrelation — a strong peak at zero lag and small values elsewhere — or the peak is ambiguous and sync lands in the wrong place. A preamble of identical repeated symbols would have a broad, flat autocorrelation and timing would be unrecoverable. This is why real standards specify particular sequences (Barker, Zadoff–Chu, m-sequences) rather than convenient bit patterns, and why every Wi-Fi packet begins with a specific preamble rather than with data.

**Now the honest limits of this demo.** The channel applies a *constant* phase offset and nothing else. Real receivers face a carrier **frequency** error — oscillator mismatch and Doppler — which makes the phase rotate continuously, so a single estimate goes stale within a few symbols and the constellation visibly spins. That is why production receivers run a phase-locked loop after acquisition rather than a one-shot correction, and why frequency offset estimation typically comes before phase correction. There is also no timing drift here (sample clocks are assumed identical) and no multipath, which Session 4 addresses.

So read "0 / 400 errors" as confirming the *principle* under favourable conditions, not as a claim that synchronisation is a solved one-liner. It is the hardest part of a real receiver, and this cell shows the first step of it.

One small note: `est_phase = ... if False else np.angle(peak)` carries a disabled branch. The alternative normalisation would only have rescaled the peak, and a positive scale factor does not change an angle — so the two are equivalent and the dead code is harmless leftover.

---
### 🕐 Session 4 of 4 — *OFDM in 40 Minutes* (~40 min)
**Goal:** beat multipath by going wide-and-slow: the FFT as a modem.
**Builds on:** Session 3; [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) S7.

---

## 5. OFDM

💡 **Intuition.** Multipath (echoes) smears fast single-carrier symbols into each other. OFDM's judo move: send **many slow streams in parallel**, one per subcarrier — and use the IFFT to pack them, the FFT to unpack. The **cyclic prefix** turns the channel's *linear* convolution into *circular* convolution ([Foundations 1 S8](./Foundations_of_Signal_Processing_1.ipynb)!), so the whole channel collapses to one complex multiply per subcarrier — equalization becomes division. Wi-Fi, LTE/5G, and DVB are exactly this.

In [ ]:

# YOUR CODE HERE


**What just happened.** The left panel is a smear — a multipath channel with echoes at 3 and 6 sample delays has scrambled the constellation past recognition. The right panel is four clean QPSK clusters. The operation between them is `Y / H`: **one complex division per subcarrier**. Symbol error rate through that hostile channel, **0.0002** — about 3 errors in 12800 symbols.

**Equalisation became division, and that is the whole point.** A wideband single-carrier link through this channel would need a long adaptive equaliser — many taps, a convergence period, ongoing tracking, and real complexity. OFDM replaces all of it with a scalar divide per subcarrier. The channel did not become easier; the *representation* changed so that the channel is diagonal in it.

**The cyclic prefix is what makes that legal.** A physical channel performs **linear** convolution, but the FFT diagonalises **circular** convolution — those are different operations, and the difference is exactly the edge effects. Copying the last 16 samples to the front makes each block look periodic to the channel across the observation window, so linear convolution *acts as* circular convolution over the part we keep. Then the convolution theorem applies and the channel collapses to one multiply per subcarrier. This is [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb)'s circular-vs-linear convolution distinction — usually a pedantic footnote — turned into the enabling trick of modern wireless.

The prefix is pure overhead: 16 samples of redundancy for every 64 of data, **25%** of the airtime, discarded on arrival. It is worth it because it deletes an entire subsystem. And it sets a hard design rule — the prefix must be **longer than the channel's delay spread**, or inter-block interference returns and the whole construction fails. That single inequality determines prefix length in Wi-Fi, LTE, and DVB.

**Why parallel-and-slow beats fast-and-equalised.** An echo 1 µs late is 300 m of extra path, entirely ordinary indoors. Against a 0.1 µs symbol that echo lands ten symbols later and smears across all of them. Make each symbol $N$ times longer by sending $N$ streams in parallel, and the same echo becomes a small fraction of one symbol. Total rate is unchanged — speed per stream traded for number of streams — and the channel becomes benign without any equaliser getting better.

The FFT is what makes it affordable: the subcarriers are orthogonal complex exponentials at multiples of a fundamental, which is precisely the DFT basis, so the IFFT *is* the modulator. $N$ oscillators would be absurd hardware; $O(N\log N)$ is a chip.

**And the honest caveats.** `H = np.fft.fft(channel, Nfft)` uses the **true** channel — perfect channel state information, which no receiver has. Real systems estimate $H$ from pilot subcarriers and pay for the estimation error. Two further omissions matter in practice: carrier **frequency offset**, to which OFDM is notoriously sensitive because it breaks subcarrier orthogonality and produces inter-carrier interference; and the high **peak-to-average power ratio** of a sum of many subcarriers, which forces amplifiers to back off and is the main practical complaint about OFDM — enough so that LTE uplinks use SC-FDMA instead. The mechanism here is real and complete; the engineering around it is where the difficulty lives.

## 6. Conclusion

Bits ride complex symbols; Nyquist pulses dodge ISI; one correlation finds both time and phase; and OFDM turns a hostile channel into $N$ trivial ones via the FFT. You've built every layer of a real modem below the error-correcting code.

---
## Where next

- [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) — the capacity these designs chase.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — run this against *real* airwaves.
- [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) — the multirate front-ends around every modem.